# Momentum baseline: reproducible offline walkthrough

A thin notebook over the phase-one engineering pipeline. It reads a **published
immutable dataset**, replays the 60-trading-day momentum spec through the CLI,
and inspects the published metrics. Everything is offline: no supplier SDK, no
network, no token, no committed market data. It is engineering validation, not
investment advice.

Point `PROJECT_ROOT` at any project that already holds a published dataset
(one produced by `python -m stock_quant data update --root <dir>`, or the
synthetic fixture built by `tests/integration/conftest.py`).

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

# Edit this to a directory that contains configs/ and data/datasets.
PROJECT_ROOT = Path(os.environ.get("STOCK_QUANT_PROJECT", ".."))
print("PROJECT_ROOT =", PROJECT_ROOT.resolve())

## 1. Open the current immutable dataset

The dataset is content-addressed and immutable: reads are pinned to one version
hash, and `CURRENT` only ever points at a fully gated, published version.

In [ ]:
from stock_quant.data_model.dataset import (
    DatasetNotFoundError,
    DatasetPublisher,
    DatasetReader,
)

try:
    version = DatasetPublisher(PROJECT_ROOT).current().version
except DatasetNotFoundError as exc:
    raise SystemExit(
        f"no published dataset under {PROJECT_ROOT.resolve()!s}. "
        "Run `python -m stock_quant data update --root <PROJECT_ROOT>` first, "
        "or build the offline fixture with tests/integration/conftest.py."
    ) from exc

print("current dataset version =", version)

with DatasetReader(PROJECT_ROOT).open(version) as context:
    daily = context.read("daily_bar")
    master = context.read("security_master")

print("daily_bar rows =", len(daily))
print("symbols =", master["symbol"].nunique())
print("date range =", daily["trade_date"].min(), "->", daily["trade_date"].max())

## 2. Reproduce the 60d momentum research run through the CLI

The CLI is the only writer of `data/experiments`. Running the same spec twice
returns the **same experiment id** with identical artifacts (content-addressed).

In [ ]:
import subprocess
import sys


def _experiment_id(text: str) -> str:
    for line in text.splitlines():
        if "experiment_id=" in line:
            return line.split("experiment_id=", 1)[1].strip()
    raise AssertionError(f"no experiment_id in output:\n{text}")


def _run(spec: str) -> str:
    result = subprocess.run(
        [sys.executable, "-m", "stock_quant", "research", "run",
         "--spec", spec, "--root", str(PROJECT_ROOT)],
        capture_output=True, text=True, check=False,
    )
    if result.returncode != 0:
        raise SystemExit(result.stdout + result.stderr)
    return result.stdout


first = _run("configs/experiments/momentum_60d.yml")
second = _run("configs/experiments/momentum_60d.yml")
print(first)
print("reproducible:", _experiment_id(first) == _experiment_id(second))

## 3. Inspect the published metrics

Each cost scenario carries the auditable engineering summary plus a richer
`PerformanceMetrics` block (`max_drawdown`, `benchmark_excess_return`, ...).

In [ ]:
import json

import pandas as pd

experiment_id = _experiment_id(first)
experiment_dir = PROJECT_ROOT / "data" / "experiments" / experiment_id
metrics = json.loads((experiment_dir / "metrics.json").read_text(encoding="utf-8"))

summary = pd.DataFrame(
    {
        scenario: {
            "end_equity": values["end_equity"],
            "total_return": values["total_return"],
            "max_drawdown": values["performance"]["max_drawdown"],
            "benchmark_excess_return": values["performance"][
                "benchmark_excess_return"
            ],
        }
        for scenario, values in metrics["scenarios"].items()
    }
).T
print(summary.to_string())
print()
print("evaluation =", metrics["evaluation"])
print("artifacts =", sorted(path.name for path in experiment_dir.iterdir()))